# Calvin Site 1 measured crown

Review a matched comparison of sampled versus field-conditioned crown positions. Each simulation constructs a fresh architecture.

In [ ]:
from pathlib import Path
from hashlib import sha256
from textwrap import dedent
import json
import os
import subprocess
import pandas as pd
from IPython.display import Image, display

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / '.runtime/calvin-site1-fit-20261119/crown20/optional_layout/PROTOCOL.json').is_file())
scripts = root / '.runtime/calvin-site1-fit-20261119'
study = scripts / 'crown20/optional_layout'
protocol = json.loads((study / 'PROTOCOL.json').read_text())
for relative, digest in protocol['basis_sha256'].items():
    assert sha256((root / relative).read_bytes()).hexdigest() == digest
metrics = pd.read_csv(study / 'scores.csv').query('split == "Site1_hold"')
display(metrics[['cutoff_mm','qc','case','spatial_score','rld_depth_tv','mass_depth_tv']].round(3))
display(pd.read_csv(study / 'indexed_crown.csv'))
display(Image(filename=str(study / 'held_core_profiles.png'), width=1010))

## Run the example

The default cells read the saved comparison. Toggle the next cell to run one local simulation from the exported schema-4 input and reproduce its two diameter-cutoff projections.

In [ ]:
RUN_EXAMPLE = False
if RUN_EXAMPLE:
    source = root / '.runtime/spatial-construction-20261119/source'
    python = root / '.runtime/envs/py310-qualification/bin/python'
    env = dict(os.environ, PYTHONPATH=str(source), PYTHONDONTWRITEBYTECODE='1',
               OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')
    command = dedent("""
        import sys
        import numpy as np
        sys.path.insert(0, sys.argv[1])
        import compare_crown_catalogue as experiment
        study = experiment.study
        engine_case = study.cutoff.read(experiment.OUT / 'Calvin_Site1_Measured_Crown.schema4.private.json')
        receipt = study.cutoff.read(experiment.OUT / 'fitted_input_replay.json')
        _, _, tubes, offsets = study.observation_support()
        simulation = study.RootSystemSimulation(random_seed=engine_case['random_seed'])
        simulation.run(engine_case)
        for cutoff in study.cutoff.CUTOFFS:
            predicted = study.cutoff.observe(simulation, cutoff, tubes, offsets).set_index('target_row')[experiment.SCORER.PREDICTIONS]
            np.testing.assert_array_equal(predicted.to_numpy(), receipt['predictions'][str(cutoff)])
        print('Both measured-crown projections reproduced.')
    """)
    subprocess.run([python,'-c',command,scripts], env=env, cwd=source, check=True)